# 01 — What's in the warehouse

A first look at the SuSSE warehouse — what tables exist, what each one
holds, and where on the map the data lives. Read this before any other
tutorial notebook. Everything downstream queries the warehouse this
notebook describes.

The notebook is **read-only**: every cell issues SELECTs only. Safe to
re-run as often as you like.

## Three things by the end

1. A mental model of the warehouse — six tables, three storage shapes,
   one catalog.
2. A map showing exactly where SuSSE has ground measurements,
   validation sites, and the inference grid.
3. Per-source coverage tables, so the next notebook
   (`02_query_the_warehouse.ipynb`) doesn't surprise you with empty
   results.

In [1]:
import sys
import os

PROJECT_ID = "solar-irradiation-estimation"

if "google.colab" in sys.modules:
    # --- GOOGLE COLAB AUTOMATED SETUP ---
    REPO = "Marconi-Lab/Solar_irradiation"
    BRANCH = "jm/add_model"
    clone_url = f"https://github.com{REPO}.git"
    
    if not os.path.exists("/content/Solar_irradiation/.git"):
        print(f"Cloning {REPO} (branch {BRANCH}) into Colab environment...")

    get_ipython().run_line_magic('cd', '/content/Solar_irradiation')
    get_ipython().system('pip install -q -e . 2>&1 | tail -3')

    from google.colab import auth
    auth.authenticate_user()
    get_ipython().system(f'gcloud config set project {PROJECT_ID} 2>/dev/null')
    print("Colab setup complete.")

else:
    # --- LOCAL LINUX AUTOMATED SETUP ---
    print("Running in Local Linux Environment.")
    
    # 1. Map paths so the notebook can find the installation
    gcloud_bin_path = os.path.expanduser("~/google-cloud-sdk/bin")
    os.environ["PATH"] = gcloud_bin_path + os.path.pathsep + os.environ["PATH"]
    os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
    
    # 2. Silently install SDK if it is missing
    if not os.path.exists(gcloud_bin_path):
        print("SDK not found. Installing now...")
        get_ipython().system('curl -sSL https://google.com | bash -s -- --disable-prompts > /dev/null')

    # 3. Trigger local system browser login directly from the notebook
    print("\nOpening your system browser for Google Cloud verification...")
    get_ipython().system('~/google-cloud-sdk/bin/gcloud auth application-default login')
    
    # 4. Set the project configuration
    get_ipython().system(f'gcloud config set project {PROJECT_ID} 2>/dev/null')
    print(f"\nLocal setup complete! Active project set to: {PROJECT_ID}")


Running in Local Linux Environment.

Opening your system browser for Google Cloud verification...
Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=ZTKgQ30DHtojtHTfUEJ5FLR6439RAK&access_type=offline&code_challenge=NS9jUp3Y4wrxigpY1wpg15Vu4mAM03KHqNf_SfDA_08&code_challenge_method=S256

Opening in existing browser session.

Credentials saved to file: [/home/rogers/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "solar-irradiation-estimation" was added to ADC which can be used by Google client librarie

## 0 — Setup

In [2]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import folium
import pandas as pd
import pygeohash
from folium import CircleMarker, FeatureGroup, LayerControl, Rectangle

# Locate project root by walking up until we see src/susse/.
_HERE = Path.cwd().resolve()
_PROJECT_ROOT = _HERE
while _PROJECT_ROOT != _PROJECT_ROOT.parent and not (_PROJECT_ROOT / "src" / "susse").exists():
    _PROJECT_ROOT = _PROJECT_ROOT.parent
sys.path.insert(0, str(_PROJECT_ROOT / "src"))

from susse.warehouse_ops.io import BigQueryClient, WarehouseConfig
from susse.warehouse_ops.io.config import TableRefs
from susse.warehouse_ops.population import (
    PhysicalStorage, Source, VariableCatalog, variables_to_dataframe,
)

bq = BigQueryClient(config=WarehouseConfig())
tables = TableRefs(config=bq.config)
print(f"warehouse: {bq.config.project_id}.{bq.config.dataset}")

## 1 — The tables, at a glance

| Table | Shape | What it stores |
|---|---|---|
| `dim_variable` | catalog | One row per `(variable_id, source)` — unit, valid range, spatial resolution, which physical table holds the value |
| `irradiance_daily` | wide | Per-`(date, geohash5, source)` daily GHI / DHI / DNI for NASA POWER and CAMS |
| `nasa_daily_vars_long` | long | NASA POWER auxiliary variables (`temperature`, `aod_550`, `cloud_amount`, …) keyed by `(date, geohash5, variable_id)` |
| `cams_daily_vars_long` | long | CAMS auxiliary variables (`ghi_clear`, `bhi`, `dni_clear`, …) |
| `merra_daily_vars_long` | long | MERRA-2 reanalysis variables (aerosol decomposition, precipitable water) |
| `modis_observations` | observation-keyed | MODIS composites keyed by `(date, geohash5, product_id, band_id)` |
| `ground_measurements` | curated long | Daily ground GHI per station with QC level |
| `ground_measurements_raw` | uncurated long | Raw partner CSVs before curation |

**Wide vs long** is chosen per source: GHI / DHI / DNI go in the wide
`irradiance_daily` because they're a tightly coupled trio always
queried together; everything else lives in a long-format companion so
adding a new variable doesn't require a schema migration.

`dim_variable.physical_storage` tags every catalog row with which of
the three storage shapes it lives in (`long_format` / `irradiance_wide`
/ `modis_observations`). `FeatureSelection` (NB 02) uses this tag to
reject typos at construction with a clear error.

For the full data lineage (which job writes which table, what's derived
from what), see [`warehouse/README.md`](../../warehouse/README.md).

### Browsing the variable catalog

Catalog: 48 variables across 4 sources.

Counts by source × storage:
physical_storage  irradiance_wide  long_format  modis_observations
source                                                            
CAMS                            3            6                   0
MERRA2                          0            4                   0
MODIS                           0            0                   3
NASA                            3           29                   0


NASA POWER (32 variables):


,variable_id,display_name,unit,physical_storage
0,ghi,All-sky GHI,kWh/m^2/day,irradiance_wide
1,dhi,All-sky DHI,kWh/m^2/day,irradiance_wide
2,dni,All-sky DNI,kWh/m^2/day,irradiance_wide
3,temperature,2m Air Temperature,degC,long_format
4,temperature_range,2m Air Temperature Range,degC,long_format
5,specific_humidity,2m Specific Humidity,kg/kg,long_format
6,relative_humidity,2m Relative Humidity,%,long_format
7,surface_pressure,Surface Pressure,kPa,long_format
8,aod_550,AOD @ 550nm,unitless,long_format
9,aod_550_adj,AOD @ 550nm (adjusted),unitless,long_format


## 2 — Coverage on the map

Three layers — toggle from the layer control in the top right.

| Marker | What it is | Source |
|---|---|---|
| **Blue dots** (sized by coverage) | Ground-measurement stations (training labels) | `ground_measurements` |
| **Red / orange / green dots** | Katongole 2023 validation stations, colour by warehouse coverage status | `data/external_references/katongole_2023_monthly.csv` + `irradiance_daily` |
| **Green rectangle** | Uganda 2024 inference grid (portal-coverage cache) | `irradiance_daily` filtered to 2024-only cells |

Click any marker for details.

/home/rogers/Github/SolarIrradiation/.venv/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Ground stations: 28


,location,lat,lon,geohash5,min_date,max_date,n_rows,n_qc_passed,n_nasa_days,n_cams_days
0,kenya_location3,-0.469974,35.181874,kzcj2,2019-05-28,2024-11-25,1976,1976,2009,2009
1,lira,2.295190,32.921370,s8rmj,2014-08-27,2023-02-01,1935,1934,3081,3081
2,kampala,0.333542,32.568630,s8p1v,2011-04-06,2023-01-22,1798,1798,4676,4676
3,ghana_location1,5.645759,-0.105223,ecpbj,2019-09-24,2024-11-25,1774,1774,1890,1890
4,nigeria_location1,9.076439,7.425385,s1t78,2020-04-04,2024-11-25,1641,1641,1697,1697


In [6]:
# Katongole validation stations + 2017-2022 coverage status.
KATONGOLE_CSV = (
    _PROJECT_ROOT / "data" / "external_references" / "katongole_2023_monthly.csv"
)
katongole = pd.read_csv(KATONGOLE_CSV)
katongole["geohash5"] = [
    pygeohash.encode(lat, lon, precision=5)
    for lat, lon in zip(katongole["latitude"], katongole["longitude"])
]
_gh_quoted = ", ".join(f"'{g}'" for g in katongole["geohash5"].unique())
cov_2017_22 = bq.query(f"""
SELECT geohash5,
       COUNT(DISTINCT IF(source='NASA', date, NULL)) AS n_nasa_days,
       COUNT(DISTINCT IF(source='CAMS', date, NULL)) AS n_cams_days
FROM `{tables.irradiance_daily}`
WHERE date BETWEEN DATE('2017-01-01') AND DATE('2022-12-31')
  AND geohash5 IN ({_gh_quoted})
GROUP BY geohash5
""")
katongole = katongole.merge(cov_2017_22, on="geohash5", how="left").fillna(
    {"n_nasa_days": 0, "n_cams_days": 0}
).astype({"n_nasa_days": int, "n_cams_days": int})
_N_EXPECTED = (pd.Timestamp("2022-12-31") - pd.Timestamp("2017-01-01")).days + 1
_FULL = int(0.95 * _N_EXPECTED)
katongole["coverage_status"] = [
    "full" if (n_nasa >= _FULL and n_cams >= _FULL)
    else "missing" if (n_nasa == 0 and n_cams == 0)
    else "partial"
    for n_nasa, n_cams in zip(katongole["n_nasa_days"], katongole["n_cams_days"])
]
print(
    f"Katongole stations: {len(katongole)} — "
    + ", ".join(
        f"{k}: {v}" for k, v in katongole['coverage_status'].value_counts().items()
    )
)

/home/rogers/Github/SolarIrradiation/.venv/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Katongole stations: 56 — full: 56


/home/rogers/Github/SolarIrradiation/.venv/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,min_lat,max_lat,min_lon,max_lon,n_cells,min_date,max_date
0,-1.5,4.21786,29.5,35.06,2183,2024-01-01,2024-12-31


## 3 — Where to go next

* [`02_query_the_warehouse.ipynb`](02_query_the_warehouse.ipynb) — build
  a `TrainingDataset` from the warehouse using `FeatureSelection` /
  `FeatureService`, plus an example of dropping down to raw SQL when
  needed.
* [`03_preprocessing.ipynb`](03_preprocessing.ipynb) → 04 → 05 — the
  rest of the pipeline: preprocessing, model factory, trainer.
* [`../../warehouse/extending_the_warehouse.ipynb`](../../warehouse/extending_the_warehouse.ipynb)
  — for contributors who need to add new variables, sources, or
  stations to the warehouse.
* [`../papers/mukiibi_mikelson_2026/01_recomputation.ipynb`](../papers/mukiibi_mikelson_2026/01_recomputation.ipynb)
  — a complete worked example, paper-faithful Random Forest GHI bias
  correction trained against the data shown above and validated against
  the Katongole network.